In [94]:
import numpy as np
from pyscf import gto, scf, cc
import jax
jax.config.update("jax_enable_x64", True)

d = 100

natom = 1
atoms = ""
for n in range(natom):
    shift = n*d
    atoms += f'O {0.0+shift} 0.0 0.0 \n'
    atoms += f'O {0.0+shift} 0.0 2.4 \n'

spin = 2
mol = gto.M(atom=atoms, 
            basis="ccpvqz", 
            spin=spin, 
            unit='B',
            verbose=4)
mol.build()

mf = scf.UHF(mol)
mf.kernel()
    
stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break


mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()

# print(mycc.energy(mycc.t1, mycc.t2*0))

System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-28-generic', version='#28~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Wed Jul  1 15:50:57 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed Aug  5 16:58:56 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      

(np.float64(-0.47455505341758003),
 (array([[-1.53932143e-08,  1.85877198e-08, -6.91939971e-09,
          -5.38232865e-10,  8.10269506e-03, -3.57335258e-03,
          -4.95576677e-09, -1.84021233e-08, -3.31852164e-09,
           1.11138983e-09, -6.27704995e-10, -7.98548486e-09,
          -4.54650678e-09, -2.23743981e-03, -2.81173856e-12,
          -4.77316986e-09, -1.71961018e-03,  1.41064199e-08,
          -1.74250407e-08,  6.83366673e-09, -3.99213783e-09,
          -2.17937176e-03, -1.79713957e-09, -4.81895218e-09,
           1.26751702e-10,  2.58788910e-10,  7.91175854e-12,
          -1.93071641e-12, -3.78497835e-09, -2.80874345e-09,
           5.10743118e-10, -2.00384835e-04,  4.59934214e-12,
           5.42406351e-12, -1.11085101e-03, -2.63970314e-10,
           7.31007363e-10,  3.23499639e-10,  2.16442958e-12,
          -1.59197977e-03, -1.85679062e-09, -1.72778833e-08,
           9.73120415e-09,  8.84825425e-13, -5.01425704e-04,
          -1.60499989e-09, -4.07189452e-09, -1.987

In [95]:
import time
import numpy as np
from jax import numpy as jnp
from jax import jit
import opt_einsum as oe

from afqmc import config
from afqmc import prep

from functools import partial
print = partial(print, flush=True)
config.setup_jax()

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      7.0.0-28-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu


In [96]:
from afqmc import integral
integral.prep_integral(mycc, chol_cut=1e-6)


Preparing AFQMC calculation
CCSD type input object
Calculating Cholesky integrals
Alpha Cholesky shape: (835, 108, 108) 
 Beta Cholesky shape: (835, 108, 108) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [7 5]
Number of basis functions:  108
Number of Cholesky vectors: 835


In [97]:
options = {'eql_time': 10,
           'n_blocks': 100,
           'n_walkers': 10,
           'mix_precision': False,
           'seed': 17,
           'guide': 'uhf',
           'trial': 'upt2ccsd_bar',
           }

In [98]:
ham_data, ham, prop, trial, wave_data, sampler, options = prep.init_afqmc(options=options)
wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)
print(mf.e_tot - prop_data["e_estimate"])


QMC Parameters
eql_time        -         10
n_blocks        -        100
n_walkers       -         10
mix_precision   -      False
seed            -         17
guide           -        uhf
trial           - upt2ccsd_bar
dt              -      0.005
n_exp_terms     -          6
n_prop_steps    -         50
walker_type     -        uhf
n_batch         -          1
max_error       -          0
nchol_chunk     -        100
max_memory      -       2000
free_projection -      False

Load system from Integral File
Maximum memory per walker:            200.00 MB
Maximum number of Cholesky per chunk: 561
Number of Cholesky chunks:            2
Number of Cholesky per chunk:         418
Number of padding Cholesky:           1

QMC System
Number of electrons: (7, 5)
Spin Multiplicity:   2
Number of orbitals:  108
Number of Chol:      835

Initalize QMC walkers by HF
7.337377780913812e-07


In [99]:
def pt2_energy_formula(h0, t2, e0, e1):
    return h0 + e0 + e1 - t2*e0

def ci_energy_formula(h0, t2, e0, e1):
    return h0 + (e0 + e1) / (1 + t2)

In [100]:
norb = trial.norb
h0 = ham_data["h0"]
h1 = ham_data["h1"]
chol = (ham_data["chol"][0].reshape(-1, norb, norb),
        ham_data["chol"][1].reshape(-1, norb, norb))

In [101]:
walker_init = (prop_data['walkers'][0][0], prop_data['walkers'][1][0])
obar, t2, e0, e1 = \
    trial._calc_energy_pt(walker_init[0], walker_init[1], ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1)-mycc.e_tot)


(-6.675771828668076e-07+0j)


In [102]:
from jax import jit, lax
import opt_einsum as oe

@partial(jit, static_argnums=0)
def _calc_energy_pt_decomposed(
    self,
    walker_up: jax.Array,
    walker_dn: jax.Array,
    ham_data: dict,
    wave_data: dict,
) -> complex:
    '''
    <bra|T2(h1+h2)|ket>/<bra|ket> with rank-decomposed T2 (be careful: tau is complex!):
        t2aa = 2 * sum_y tau_a[y,i,a] tau_a[y,j,b]
        t2ab =     sum_y tau_a[y,i,a] tau_b[y,j,b]
        t2bb = 2 * sum_y tau_b[y,i,a] tau_b[y,j,b]
    '''
    if self.mix_precision:
        rtype = jnp.float32
        ctype = jnp.complex64
    else:
        rtype = jnp.float64
        ctype = jnp.complex128

    norb_a, nocc_a = walker_up.shape
    norb_b, nocc_b = walker_dn.shape

    tau_a = wave_data["tau_a"]          # (ny, nocc_a, nvir_a)
    tau_b = wave_data["tau_b"]          # (ny, nocc_b, nvir_b)

    chol_a = ham_data["chol_bar"][0]
    chol_b = ham_data["chol_bar"][1]
    h1_a = ham_data["h1_bar"][0]
    h1_b = ham_data["h1_bar"][1]

    walker_up = wave_data['exp_t1a'] @ walker_up
    walker_dn = wave_data['exp_t1b'] @ walker_dn

    green_a = (walker_up.dot(jnp.linalg.inv(walker_up[:nocc_a, :]))).T
    green_b = (walker_dn.dot(jnp.linalg.inv(walker_dn[:nocc_b, :]))).T
    greenov_a = green_a[:nocc_a, nocc_a:]
    greenov_b = green_b[:nocc_b, nocc_b:]
    greenp_a = jnp.vstack((greenov_a, -jnp.eye(norb_a - nocc_a)))
    greenp_b = jnp.vstack((greenov_b, -jnp.eye(norb_b - nocc_b)))

    hg_a = oe.contract("pq,pq->", h1_a[:nocc_a, :], green_a, backend="jax")
    hg_b = oe.contract("pq,pq->", h1_b[:nocc_b, :], green_b, backend="jax")
    e1_0 = hg_a + hg_b  # <bra|h1|ket>/<bra|ket>

    # ---- <bra|T2 h1|ket>: scan over the decomposition rank y ----
    def scan_tau1(carry, x):
        tau_a_y, tau_b_y = x
        # spin a
        taug_a   = oe.contract("ia,ja->ij", tau_a_y, greenov_a, backend="jax")
        tr_a     = oe.contract("ii->", taug_a, backend="jax")
        taugp_a  = oe.contract("jb,pb->jp", tau_a_y, greenp_a, backend="jax")
        taugpg_a = oe.contract("jp,jq->pq", taugp_a, green_a[:nocc_a, :], backend="jax")
        # spin b
        taug_b   = oe.contract("ia,ja->ij", tau_b_y, greenov_b, backend="jax")
        tr_b     = oe.contract("ii->", taug_b, backend="jax")
        taugp_b  = oe.contract("jb,pb->jp", tau_b_y, greenp_b, backend="jax")
        taugpg_b = oe.contract("jp,jq->pq", taugp_b, green_b[:nocc_b, :], backend="jax")

        carry[0] += tr_a ** 2 + tr_b ** 2 + tr_a * tr_b   # gt2g   = <bra|T2|ket>
        carry[1] += (2 * tr_a + tr_b) * taugpg_a          # t2_green_a_a (4*aaa + aba)
        carry[2] += (2 * tr_b + tr_a) * taugpg_b          # t2_green_b_b (4*bbb + abb)
        return carry, 0.0

    init = [jnp.zeros((), jnp.complex128),                 # gt2g          (scalar)
            jnp.zeros((norb_a, norb_a), jnp.complex128),   # t2_green_a_a  (matrix)
            jnp.zeros((norb_b, norb_b), jnp.complex128)]   # t2_green_b_b  (matrix)
    [gt2g, t2_green_a_a, t2_green_b_b], _ = lax.scan(scan_tau1, init, (tau_a, tau_b))

    e1_2_1 = e1_0 * gt2g
    e1_2_2_a = -oe.contract("pq,pq->", h1_a, t2_green_a_a, backend="jax")
    e1_2_2_b = -oe.contract("pq,pq->", h1_b, t2_green_b_b, backend="jax")
    e1_2 = e1_2_1 + e1_2_2_a + e1_2_2_b  # <bra|T2 h1|ket>/<bra|ket>

    # ---- <bra|T2 h2|ket>: scan over cholesky chunks ----
    nchol = chol_a.shape[0]
    nchunks = -(-nchol // self.nchol_chunk)
    pad = nchunks * self.nchol_chunk - nchol
    chol_a = jnp.pad(chol_a, ((0, pad), (0, 0), (0, 0)))
    chol_b = jnp.pad(chol_b, ((0, pad), (0, 0), (0, 0)))
    chol_a = chol_a.reshape(nchunks, self.nchol_chunk, *chol_a.shape[-2:])
    chol_b = chol_b.reshape(nchunks, self.nchol_chunk, *chol_b.shape[-2:])

    def scanned_fun(carry, x):
        chol_a_c, chol_b_c = x

        # e2_0 = <h2>   (no T2 — identical to the dense function)
        gl_a_c = oe.contract("ir,gpr->gip",
                                green_a.astype(jnp.complex128),
                                chol_a_c.astype(jnp.float64),
                                backend="jax").astype(jnp.complex128)
        gl_b_c = oe.contract("ir,gpr->gip",
                                green_b.astype(jnp.complex128),
                                chol_b_c.astype(jnp.float64),
                                backend="jax")
        tr_gl_a = oe.contract("gii->g", gl_a_c[:, :nocc_a, :nocc_a], backend="jax").astype(jnp.complex128)
        tr_gl_b = oe.contract("gii->g", gl_b_c[:, :nocc_b, :nocc_b], backend="jax").astype(jnp.complex128)
        ex_gl_a = oe.contract("gij,gji->g", gl_a_c[:, :nocc_a, :nocc_a], gl_a_c[:, :nocc_a, :nocc_a], backend="jax").astype(jnp.complex128)
        ex_gl_b = oe.contract("gij,gji->g", gl_b_c[:, :nocc_b, :nocc_b], gl_b_c[:, :nocc_b, :nocc_b], backend="jax").astype(jnp.complex128)
        e2_0_1_c = jnp.sum((tr_gl_a + tr_gl_b) ** 2) / 2.0
        e2_0_2_c = -jnp.sum(ex_gl_a + ex_gl_b) / 2.0
        carry[0] += (e2_0_1_c + e2_0_2_c).astype(jnp.complex128)

        # e2_2_2 = <T2 h2> pieces that only need t2_green (no T2 directly)
        lt2g_a_c = oe.contract("gpr,qr->gpq", chol_a_c.astype(jnp.float64),
                                (2 * t2_green_a_a).astype(jnp.complex128), backend="jax")
        lt2g_b_c = oe.contract("gpr,qr->gpq", chol_b_c.astype(jnp.float64),
                                (2 * t2_green_b_b).astype(jnp.complex128), backend="jax")
        tr_lt2g_a_c = oe.contract("gqq->g", lt2g_a_c, backend="jax")
        tr_lt2g_b_c = oe.contract("gqq->g", lt2g_b_c, backend="jax")
        carry[1] += -(((tr_lt2g_a_c.astype(ctype) + tr_lt2g_b_c.astype(ctype))
                        @ (tr_gl_a.astype(ctype) + tr_gl_b.astype(ctype))) / 2).astype(jnp.complex128)
        carry[2] += ((oe.contract("giq,giq->", gl_a_c.astype(ctype), lt2g_a_c[:, :nocc_a, :].astype(ctype), backend="jax")
                    + oe.contract("giq,giq->", gl_b_c.astype(ctype), lt2g_b_c[:, :nocc_b, :].astype(ctype), backend="jax")) / 2).astype(jnp.complex128)

        # e2_2_3 = decomposed T2 two-body — inner scan over y
        glgp_a_c = oe.contract("giq,qa->gia", gl_a_c, greenp_a.astype(jnp.complex128), backend="jax")
        glgp_b_c = oe.contract("giq,qa->gia", gl_b_c, greenp_b.astype(jnp.complex128), backend="jax")

        def scan_tau2(carry, x):
            tau_a_y, tau_b_y = x
            # A_s[g] = sum_ia glgp_s[g,i,a] tau_s[y,i,a]
            A_a = oe.contract("gia,ia->g", glgp_a_c.astype(ctype), tau_a_y.astype(ctype), backend="jax")
            A_b = oe.contract("gia,ia->g", glgp_b_c.astype(ctype), tau_b_y.astype(ctype), backend="jax")
            carry[0] += oe.contract("g,g->", A_a, A_a, backend="jax").astype(jnp.complex128)  # l2t2_aa
            carry[1] += oe.contract("g,g->", A_b, A_b, backend="jax").astype(jnp.complex128)  # l2t2_bb
            carry[2] += oe.contract("g,g->", A_a, A_b, backend="jax").astype(jnp.complex128)  # l2t2_ab
            return carry, 0.0

        init2 = [jnp.zeros((), jnp.complex128),
                 jnp.zeros((), jnp.complex128),
                 jnp.zeros((), jnp.complex128)]
        [l2t2_aa, l2t2_bb, l2t2_ab], _ = lax.scan(scan_tau2, init2, (tau_a, tau_b))
        carry[3] += (l2t2_aa + l2t2_bb + l2t2_ab).astype(jnp.complex128)
        return carry, 0.0

    init_c = [jnp.zeros((), jnp.complex128)] * 4
    [e2_0, e2_2_2_1, e2_2_2_2, e2_2_3], _ = lax.scan(
        scanned_fun, init_c, (chol_a, chol_b)
    )

    e2_2_1 = e2_0 * gt2g
    e2_2_2 = e2_2_2_1 + e2_2_2_2
    e2_2 = e2_2_1 + e2_2_2 + e2_2_3  # <bra|T2 h2|ket>/<bra|ket>

    ot1 = jnp.linalg.det(walker_up[:nocc_a, :]) \
        * jnp.linalg.det(walker_dn[:nocc_b, :])  # <bra|ket_bar>
    t2 = gt2g
    e0 = e1_0 + e2_0
    e1 = e1_2 + e2_2
    return ot1, t2, e0, e1

In [112]:
from afqmc import t2_tools
wave_data["tau_a"], wave_data["tau_b"] \
      = t2_tools.decompose_t2((wave_data["t2aa"],
                               wave_data["t2ab"],
                               wave_data["t2bb"]),
                               1e-1)
print(f"rank reduction: "
      f"{wave_data["tau_a"].shape[1]*wave_data["tau_a"].shape[2]} -> {wave_data['tau_a'].shape[0]}  "
      f"{wave_data["tau_b"].shape[1]*wave_data["tau_b"].shape[2]} -> {wave_data['tau_b'].shape[0]}  ")

rank reduction: 707 -> 3  515 -> 3  


In [113]:
t2aa_rec = oe.contract('gia,gjb->iajb', wave_data["tau_a"], wave_data["tau_a"], backend='jax')
t2ab_rec = oe.contract('gia,gjb->iajb', wave_data["tau_a"], wave_data["tau_b"], backend='jax')
t2bb_rec = oe.contract('gia,gjb->iajb', wave_data["tau_b"], wave_data["tau_b"], backend='jax')
print(abs(t2aa_rec * 2 - wave_data["t2aa"]).max()) # numerical noise
print(abs(t2ab_rec - wave_data["t2ab"]).max()) # numerical noise
print(abs(t2bb_rec * 2 - wave_data["t2bb"]).max()) # numerical noise

0.015279055289697361
0.017936050722395853
0.08102694058456583


In [114]:
walker_up = jnp.array(np.random.rand(*walker_init[0].shape))
walker_dn = jnp.array(np.random.rand(*walker_init[1].shape))

obar, t2, e0, e1 = \
    trial._calc_energy_pt(walker_up, walker_dn, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1))
print(ci_energy_formula(h0, t2, e0, e1))


obar, t2, e0, e1 = \
    _calc_energy_pt_decomposed(trial, walker_up, walker_dn, ham_data, wave_data)
print(pt2_energy_formula(h0, t2, e0, e1))
print(ci_energy_formula(h0, t2, e0, e1))

# obar, t2, e0, e1 = \
#     _calc_energy_pt2_decomposed_new(trial, walker, ham_data, wave_data)
# print(pt2_energy_formula(h0, t2, e0, e1))

(-177.93858021254846+0j)
(-151.60572137068812+0j)
(-665.6813000368384+0j)
(632.8480623063354+0j)


In [115]:
import time
import numpy as np

try:
    import jax
    _HAS_JAX = True
except ImportError:
    _HAS_JAX = False


def benchmark(fn, n_warmup=3, n_runs=30, label=""):
    """Time a zero-arg callable. Handles JAX JIT warm-up + async dispatch."""
    # Warm-up: triggers JIT compile; NOT timed
    for _ in range(n_warmup):
        out = fn()
        if _HAS_JAX:
            jax.block_until_ready(out)

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        out = fn()
        if _HAS_JAX:
            jax.block_until_ready(out)   # wait for the real compute to finish
        times.append(time.perf_counter() - t0)

    t = np.asarray(times)
    print(f"{label:20s} {t.mean()*1e3:8.3f} ± {t.std()*1e3:6.3f} ms "
          f"(min {t.min()*1e3:8.3f} ms, n={n_runs})")
    return t.mean(), t.min()


# the two calls as zero-arg thunks
old = lambda: trial._calc_energy_pt(walker_up, walker_dn, ham_data, wave_data)
new = lambda: _calc_energy_pt_decomposed(trial, walker_up, walker_dn, ham_data, wave_data)
# new = lambda: _calc_energy_pt2_decomposed(trial, walker, ham_data, wave_data)

# sanity: confirm they still agree before timing
o1, o2 = old(), new()
e_old = pt2_energy_formula(h0, o1[1], o1[2], o1[3])
e_new = pt2_energy_formula(h0, o2[1], o2[2], o2[3])
print(f"energy old={e_old}, new={e_new}, diff={abs(e_old - e_new):.3e}\n")

# benchmark
mean_old, min_old = benchmark(old, label="old (dense T2):")
mean_new, min_new = benchmark(new, label="new (decomposed):")

print(f"\nspeedup (mean): {mean_old/mean_new:.2f}x")
print(f"speedup (min):  {min_old/min_new:.2f}x")

energy old=(-177.93858021254846+0j), new=(-665.6813000368384+0j), diff=4.877e+02

old (dense T2):        76.061 ±  0.021 ms (min   75.967 ms, n=30)
new (decomposed):      56.445 ±  0.073 ms (min   56.392 ms, n=30)

speedup (mean): 1.35x
speedup (min):  1.35x
